# IOAI — 2025 Contest Cuties Segmentation (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
os.system('pip -q install scikit-image')
if not os.path.exists('data/val_imgs'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-cuties-segmentation', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
    # 캐글 zip 은 data/cuties/{val_imgs,val_masks,test_imgs} 구조 → data/ 로 평탄화
    if os.path.isdir('data/cuties'):
        for d in ['val_imgs','val_masks','test_imgs']:
            if os.path.isdir(f'data/cuties/{d}') and not os.path.isdir(f'data/{d}'): os.rename(f'data/cuties/{d}', f'data/{d}')
print('데이터 준비:', 'data/val_imgs' if os.path.isdir('data/val_imgs') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Cuties Segmentation · 모범답안 (Otsu + 후처리)

베이스라인(평균 임계값, IoU ≈0.355)을 개선합니다. 히트맵→마스크 단계만 바꿉니다(CLIP 패치 유사도는 동일):
1. **Otsu 임계값** — 전경/배경 이중분포에 최적인 임계값(평균보다 정확)
2. **최대 연결요소만 유지 + 구멍 채움** — 잡음 영역 제거, 동물 한 덩어리로 정리

검증 평균 IoU: 0.355 → **0.417**. (참고: "고양이/개 vs 배경" 대조 프롬프트는 16×16 작은 패치엔 오히려 해로워서
*선택 품종* 유사도를 그대로 사용합니다. 리더보드 1위 ≈0.738.)

### ⚙️ 먼저 실행 — transformers 버전 고정
이 CLIP 제로샷 분할은 **transformers 4.51.x** 에서 검증된 결과입니다(5.x 는 동일 코드라도 CLIP 임베딩이 달라져 IoU 가 떨어지고 `get_text_features` 반환형도 바뀜). 대회/레퍼런스 기준 버전으로 고정합니다. `--no-deps` 로 다른 패키지 의존성 경고 없이 transformers 만 교체합니다(설치 후 커널이 새 버전 사용).

In [ ]:
import subprocess, sys
# 대회 검증 버전(transformers 4.51.x)을 격리 디렉토리에 설치해 sys.path 최우선으로 로드.
# 4.51 과 호환되는 tokenizers(<0.22)·huggingface_hub(<1.0)도 함께 격리 설치(시스템 5.x 패키지는 그대로).
if '/tmp/tf451' not in sys.path:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location',
                    '--target', '/tmp/tf451', '--no-deps',
                    'transformers==4.51.3', 'tokenizers==0.21.1', 'huggingface_hub==0.30.2'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    sys.path.insert(0, '/tmp/tf451')
import transformers; print('transformers', transformers.__version__)

In [ ]:
import os, numpy as np, torch
from torch.nn import functional as F
from PIL import Image
from transformers import AutoProcessor, CLIPModel
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
mn = "openai/clip-vit-base-patch16"
proc = AutoProcessor.from_pretrained(mn); model = CLIPModel.from_pretrained(mn).to(dev).eval()
class_names = ['american_bulldog','basset_hound','keeshond','British_Shorthair','Sphynx','pomeranian','Egyptian_Mau','Birman','american_pit_bull_terrier','japanese_chin','Maine_Coon','beagle','Bombay','wheaten_terrier','shiba_inu','havanese','miniature_pinscher','yorkshire_terrier','great_pyrenees','leonberger','english_cocker_spaniel','scottish_terrier','Abyssinian','Ragdoll','Persian','staffordshire_bull_terrier','chihuahua','newfoundland','german_shorthaired','samoyed','english_setter','boxer','Russian_Blue','saint_bernard','pug','Bengal']
PS, SZ = 16, 224; NP = SZ // PS
# transformers 4.x/5.x 호환: get_text_features 가 5.x 에선 텐서가 아닌 출력 객체를 돌려줘 직접 투영한다
@torch.no_grad()
def txt(ts):
    t = proc(text=ts, images=None, return_tensors="pt", padding=True).to(dev)
    out = model.get_text_features(**t)
    if not torch.is_tensor(out): out = model.text_projection(out.pooler_output)
    return F.normalize(out, dim=-1)
@torch.no_grad()
def emb(imgs):
    inp = proc(text='', images=imgs, return_tensors="pt").to(dev)
    out = model(**inp)
    ie = out.image_embeds if (hasattr(out,'image_embeds') and torch.is_tensor(getattr(out,'image_embeds',None))) else model.visual_projection(model.vision_model(pixel_values=inp['pixel_values']).pooler_output)
    return F.normalize(ie, dim=-1)
CE = txt(class_names)            # 클래스(품종) 텍스트 임베딩
VAL = 'data/val_imgs'
def heatmap_for(img):
    """이미지 → 선택 클래스 기준 CLIP 패치 유사도 히트맵(원본 크기, 0~1)."""
    sh = np.array(img).shape
    cls = (emb(img) @ CE.T).argmax(1)[0]                  # 전체 이미지로 클래스 선택
    a = np.array(img.resize((SZ, SZ)))
    patches = [Image.fromarray(a[x:x+PS, y:y+PS]) for x in range(0,SZ,PS) for y in range(0,SZ,PS)]
    sims = (emb(patches) @ CE[cls].unsqueeze(0).T).reshape(NP, NP)
    hm = torch.nn.functional.interpolate(sims[None,None].float(), scale_factor=PS, mode='bilinear')[0,0].cpu().numpy()
    hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)
    return hm, sh

In [ ]:
from scipy import ndimage
def otsu(hm, bins=256):
    hist, edges = np.histogram(hm.ravel(), bins=bins, range=(0.0, 1.0))
    centers = (edges[:-1] + edges[1:]) / 2
    w0 = hist.cumsum().astype(float); total = w0[-1]; w1 = total - w0
    cs = (hist * centers).cumsum()
    m0 = np.where(w0 > 0, cs / np.maximum(w0, 1), 0)
    m1 = np.where(w1 > 0, (cs[-1] - cs) / np.maximum(w1, 1), 0)
    between = w0 * w1 * (m0 - m1) ** 2
    return centers[int(np.argmax(between))]
def postprocess(m):
    m = ndimage.binary_fill_holes(m)
    lbl, n = ndimage.label(m)
    if n > 1:                                  # 가장 큰 연결요소만 남김
        sizes = ndimage.sum(m, lbl, range(1, n+1)); m = (lbl == (np.argmax(sizes)+1))
    return m.astype(np.uint8)

masks = {}
for n in sorted(os.listdir(VAL)):
    img = Image.open(os.path.join(VAL, n)).convert('RGB')
    hm, sh = heatmap_for(img)
    m = (hm >= otsu(hm)).astype(np.uint8)        # Otsu 임계값
    m = np.array(Image.fromarray(m*255).resize((sh[1], sh[0]), Image.NEAREST)) // 255
    m = postprocess(m)                                     # 최대 연결요소 + 구멍 채움
    masks[n.replace('.jpg','')] = m.astype(np.uint8)
np.savez_compressed('submission.npz', **masks)
print('wrote submission.npz', len(masks), 'masks')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npz']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)